# Map layers — review

Plots every downloaded map layer so each can be checked by eye before any of them is used.
Nothing here filters, joins, or modifies the data; it only loads and draws it.

Data comes from `scripts/fetch_layers.py` → `data/raw/layers/`. Run that first.
Kernel: **Chicago Council** (`conda env create -f environment.yml`).

In [ ]:
from pathlib import Path
import json

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import contextily

# --- Settings (every choice made in this notebook) --------------------------

LAYERS_DIR = Path("../data/raw/layers")

# Coordinate system for plotting. Web Mercator is what basemap tiles use.
PLOT_CRS = "EPSG:3857"

# Street basemap. Draws tiles fetched live from the provider (needs internet).
# Tested 2026-09-29: CARTO tiles now require an API key, and OpenStreetMap's
# tile servers block contextily's requests ("Access blocked"). Both Esri
# options below work without a key.
DRAW_BASEMAP = True
BASEMAP_OPTIONS = {
    "street": contextily.providers.Esri.WorldStreetMap,   # colored, street names, parks
    "gray": contextily.providers.Esri.WorldGrayCanvas,    # light gray, minimal labels
}
BASEMAP_CHOICE = "street"   # street or gray change to "gray" when colored overlays are hard to read
BASEMAP_TILES = BASEMAP_OPTIONS[BASEMAP_CHOICE]

# Ward outline drawn over every layer for reference.
WARD_OUTLINE_COLOR = "black"
WARD_OUTLINE_WIDTH = 0.6

# The ward used for the zoomed-in check at the end (47th = Ald. Martin).
ZOOM_WARD = "47"

# Rows of attribute data printed per layer.
SAMPLE_ROWS = 3

FIGURE_SIZE_SINGLE = (9, 11)


## Manifest — what was downloaded, from where, and when

In [ ]:
manifest = json.loads((LAYERS_DIR / "manifest.json").read_text())
manifest_table = pd.DataFrame(manifest).T[
    ["dataset_id", "feature_count", "bytes", "retrieved_at", "portal_page", "notes"]
]
manifest_table["megabytes"] = (manifest_table.pop("bytes") / 1e6).round(2)
pd.set_option("display.max_colwidth", 120)
manifest_table

## Load every layer
For each: geometry types, coordinate system, columns, and a few sample rows.

In [ ]:
layers = {}
for layer_name in manifest:
    layers[layer_name] = gpd.read_file(LAYERS_DIR / f"{layer_name}.geojson")

for layer_name, frame in layers.items():
    print(f"=== {layer_name}: {len(frame)} rows, CRS={frame.crs}")
    print("geometry types:", frame.geom_type.value_counts().to_dict())
    print("empty or missing geometries:", int((frame.geometry.isna() | frame.geometry.is_empty).sum()))
    print("invalid geometries:", int((~frame.geometry.is_valid).sum()))
    print("columns:", list(frame.columns))
    display(frame.drop(columns="geometry").head(SAMPLE_ROWS))

## Overview grid — each layer alone, with ward outlines

In [ ]:
wards_plot = layers["wards_2023"].to_crs(PLOT_CRS)

column_count = 4
row_count = -(-len(layers) // column_count)  # ceiling division
figure, axes_grid = plt.subplots(row_count, column_count, figsize=(4 * column_count, 5 * row_count))
for axis, (layer_name, frame) in zip(axes_grid.flat, layers.items()):
    frame.to_crs(PLOT_CRS).plot(ax=axis, color="tab:blue", alpha=0.5, edgecolor="tab:blue", linewidth=0.3, markersize=4)
    wards_plot.boundary.plot(ax=axis, color=WARD_OUTLINE_COLOR, linewidth=0.3)
    axis.set_title(f"{layer_name} (n={len(frame)})", fontsize=10)
    axis.set_axis_off()
for unused_axis in list(axes_grid.flat)[len(layers):]:
    unused_axis.set_axis_off()
plt.tight_layout()

## One large map per layer — layer shaded and labeled, ward outlines + numbers on top

**Polygon layers** (community areas, neighborhoods, etc.) are filled using a small palette so that
no two neighboring shapes share a color, and labeled with their names where that is readable.
**Point and line layers** are drawn as markers / lines. Every map then draws ward outlines and ward
numbers on top, over the street basemap.

How the coloring works (so it can be checked): two shapes count as *neighbors* if they come within
`NEIGHBOR_DISTANCE_METERS` of each other (a small tolerance, because adjacent boundaries in these
files don't always meet exactly). Shapes are colored one at a time, most-neighbors first, each
taking the first palette color none of its already-colored neighbors uses. This usually needs 4–5
colors; the number actually used is printed for each layer. (A 4-color result is always *possible*
for a map, but this simple method doesn't guarantee finding it.)

In [ ]:
# --- Styling settings for the per-layer maps ---------------------------------

# How each layer is drawn and which column (if any) labels it.
#   "fill"   = polygon layer, neighbor-aware multi-color fill
#   "single" = polygon layer, one color (used for parks: scattered, not a tiling of the city)
#   "line" / "point" = drawn as lines / markers
# label_column = None means no labels (e.g. ~800 census tracts are too dense to label legibly).
LAYER_STYLE = {
    "wards_2023":         {"kind": "fill",   "label_column": None},  # ward numbers are drawn anyway
    "community_areas":    {"kind": "fill",   "label_column": "community"},
    "neighborhoods":      {"kind": "fill",   "label_column": "pri_neigh"},
    "census_tracts_2020": {"kind": "fill",   "label_column": None},
    "precincts_2023":     {"kind": "fill",   "label_column": None},
    "zip_codes":          {"kind": "fill",   "label_column": "zip"},
    "police_districts":   {"kind": "fill",   "label_column": "dist_num"},
    "parks":              {"kind": "single", "label_column": None},
    "cta_rail_lines":     {"kind": "line",   "label_column": None},
    "cta_rail_stations":  {"kind": "point",  "label_column": None},
    "cps_schools_sy2526": {"kind": "point",  "label_column": None},
    "ward_offices":       {"kind": "point",  "label_column": None},
}

# Fill palette: soft colors so the basemap and ward lines stay visible. Extra
# entries beyond four are there in case the simple coloring method needs them.
FILL_PALETTE = ["#8dd3c7", "#ffffb3", "#bebada", "#fb8072", "#80b1d3", "#fdb462", "#b3de69"]
FILL_OPACITY = 0.45
SINGLE_FILL_COLOR = "#2ca25f"      # parks
LINE_COLOR = "#6a3d9a"             # CTA rail lines
POINT_COLOR = "#d7301f"            # stations, schools, ward offices
POINT_SIZE = 14

# Two shapes are "neighbors" (must get different colors) if closer than this.
NEIGHBOR_DISTANCE_METERS = 5

LAYER_LABEL_FONT_SIZE = 6
LAYER_LABEL_COLOR = "#333333"
WARD_NUMBER_FONT_SIZE = 9
WARD_OUTLINE_WIDTH_LARGE_MAP = 1.4
FIGURE_SIZE_LARGE = (12, 15)


In [ ]:
from matplotlib import patheffects

# White outline around text so labels stay readable over any fill color.
TEXT_HALO = [patheffects.withStroke(linewidth=2.5, foreground="white")]


def neighbor_color_indexes(frame_in_meters):
    """Assign each shape a palette index so that neighboring shapes differ.

    Greedy coloring, most-neighbors-first. Returns a list of indexes aligned
    with the rows of frame_in_meters.
    """
    shapes = frame_in_meters.geometry.reset_index(drop=True)
    grown_shapes = shapes.buffer(NEIGHBOR_DISTANCE_METERS)
    spatial_index = shapes.sindex
    neighbors_of = {}
    for row_number, grown_shape in enumerate(grown_shapes):
        candidate_rows = spatial_index.query(grown_shape, predicate="intersects")
        neighbors_of[row_number] = {int(other) for other in candidate_rows if other != row_number}

    color_of = {}
    for row_number in sorted(neighbors_of, key=lambda row: len(neighbors_of[row]), reverse=True):
        colors_taken = {color_of[other] for other in neighbors_of[row_number] if other in color_of}
        color_of[row_number] = next(index for index in range(len(FILL_PALETTE) + 100) if index not in colors_taken)
    return [color_of[row_number] for row_number in range(len(shapes))]


def draw_layer(layer_name, frame, axis):
    """Draw one layer according to LAYER_STYLE. Returns a short note for the title."""
    style = LAYER_STYLE[layer_name]
    frame_plot = frame.to_crs(PLOT_CRS)
    note = ""
    if style["kind"] == "fill":
        color_indexes = neighbor_color_indexes(frame_plot)
        colors_used = max(color_indexes) + 1
        if colors_used > len(FILL_PALETTE):
            print(f"WARNING {layer_name}: needed {colors_used} colors, palette has {len(FILL_PALETTE)}; colors repeat")
        fill_colors = [FILL_PALETTE[index % len(FILL_PALETTE)] for index in color_indexes]
        frame_plot.plot(ax=axis, color=fill_colors, alpha=FILL_OPACITY, edgecolor="#777777", linewidth=0.4)
        note = f", {colors_used} fill colors"
    elif style["kind"] == "single":
        frame_plot.plot(ax=axis, color=SINGLE_FILL_COLOR, alpha=0.6, edgecolor=SINGLE_FILL_COLOR, linewidth=0.4)
    elif style["kind"] == "line":
        frame_plot.plot(ax=axis, color=LINE_COLOR, linewidth=2)
    elif style["kind"] == "point":
        frame_plot.plot(ax=axis, color=POINT_COLOR, markersize=POINT_SIZE, edgecolor="white", linewidth=0.5)

    if style["label_column"]:
        for label_text, label_point in zip(frame_plot[style["label_column"]], frame_plot.representative_point()):
            axis.annotate(str(label_text), (label_point.x, label_point.y), fontsize=LAYER_LABEL_FONT_SIZE,
                          color=LAYER_LABEL_COLOR, ha="center", va="center", style="italic",
                          path_effects=TEXT_HALO, annotation_clip=True)
    return note


def draw_wards_on_top(axis):
    wards_plot.boundary.plot(ax=axis, color=WARD_OUTLINE_COLOR, linewidth=WARD_OUTLINE_WIDTH_LARGE_MAP)
    for ward_number, label_point in zip(wards_plot["ward"], wards_plot.representative_point()):
        axis.annotate(ward_number, (label_point.x, label_point.y), fontsize=WARD_NUMBER_FONT_SIZE,
                      ha="center", va="center", weight="bold", path_effects=TEXT_HALO, annotation_clip=True)


In [ ]:
for layer_name, frame in layers.items():
    figure, axis = plt.subplots(figsize=FIGURE_SIZE_LARGE)
    note = draw_layer(layer_name, frame, axis)
    draw_wards_on_top(axis)
    if DRAW_BASEMAP:
        contextily.add_basemap(axis, source=BASEMAP_TILES, crs=PLOT_CRS)
    axis.set_title(f"{layer_name} (n={len(frame)}{note}) — ward outlines and numbers on top")
    axis.set_axis_off()
    plt.show()

## Zoom: every layer around one ward

Area shown is the bounding box of `ZOOM_WARD`. Layers are drawn unclipped, so features crossing the
ward line are visible.

In [ ]:
zoom_ward_shape = wards_plot[wards_plot["ward"] == ZOOM_WARD]
x_min, y_min, x_max, y_max = zoom_ward_shape.total_bounds
margin = 0.1 * max(x_max - x_min, y_max - y_min)

for layer_name, frame in layers.items():
    if layer_name == "wards_2023":
        continue
    figure, axis = plt.subplots(figsize=(8, 8))
    note = draw_layer(layer_name, frame, axis)
    draw_wards_on_top(axis)
    zoom_ward_shape.boundary.plot(ax=axis, color=WARD_OUTLINE_COLOR, linewidth=3)
    axis.set_xlim(x_min - margin, x_max + margin)
    axis.set_ylim(y_min - margin, y_max + margin)
    if DRAW_BASEMAP:
        contextily.add_basemap(axis, source=BASEMAP_TILES, crs=PLOT_CRS)
    axis.set_title(f"Ward {ZOOM_WARD} — {layer_name}{note}")
    axis.set_axis_off()
    plt.show()